# [실습 4-1] 미니 전방 연쇄 추론 엔진 — 동물 분류 규칙 10개

| 항목 | 내용 |
|---|---|
| 예상 소요 시간 | 20~30분 (CPU 충분) |
| 본문 연계 | 4.4 규칙 기반 시스템 |
| 선수 실습 | 없음 (Python 기본 문법만 사용) |
| 준비 | 부록 B.1(Colab 시작)·B.2(Python 문법) 참고 |

지식 베이스(규칙 10개)와 추론 엔진을 **분리**해서 만들고,
본문 4.4 IF-THEN 추론에서 손으로 추적한 유도 과정을 코드로 재현한다.

### [준비] 환경 확인 (저장소 전용)\n\n이 실습은 **표준 라이브러리만** 사용한다 — 설치할 것이 없다.

In [1]:
import platform
print("Python", platform.python_version())
print("외부 라이브러리 불요 — 바로 시작한다.")

Python 3.12.6
외부 라이브러리 불요 — 바로 시작한다.


### [셀 1] 지식 베이스 — 동물 분류 규칙 10개 📖

> 동물 분류 규칙은 인공지능 교육의 고전 예제(Winston, *Artificial Intelligence*, 3rd ed., 1992, 7장)를 다듬은 것이다.

In [2]:
# 지식 베이스: 규칙 = "조건들(if)이 모두 참이면 결론(then)"
RULES = {
    "R1": {"if": ["털이 있다"], "then": "포유류"},
    "R2": {"if": ["우유를 먹인다"], "then": "포유류"},
    "R3": {"if": ["깃털이 있다"], "then": "조류"},
    "R4": {"if": ["날 수 있다", "알을 낳는다"],
           "then": "조류"},
    "R5": {"if": ["포유류", "고기를 먹는다"],
           "then": "육식동물"},
    "R6": {"if": ["포유류", "송곳니가 있다",
                  "발톱이 있다"], "then": "육식동물"},
    "R7": {"if": ["포유류", "발굽이 있다"],
           "then": "유제류"},
    "R8": {"if": ["육식동물", "황갈색이다",
                  "검은 줄무늬가 있다"], "then": "호랑이"},
    "R9": {"if": ["육식동물", "황갈색이다",
                  "검은 점무늬가 있다"], "then": "치타"},
    "R10": {"if": ["유제류", "목이 길다", "다리가 길다"],
            "then": "기린"},
}
print(f"규칙 {len(RULES)}개 준비 완료")

규칙 10개 준비 완료


**핵심 포인트**
- 지식이 **데이터**(딕셔너리)로 표현되어 있다 — 규칙을 고쳐도 엔진 코드는 그대로다(4.4의 지식 베이스·추론 엔진 분리).
- 규칙의 결론이 다른 규칙의 조건이 된다(R1의 "포유류" → R5의 조건) — 연쇄 추론의 재료.

기대 출력: `규칙 10개 준비 완료`

### [셀 2] 추론 엔진 — 전방 연쇄 📖

In [3]:
def forward_chain(rules, initial_facts):
    """새 사실이 나오지 않을 때까지 규칙을 반복 적용."""
    facts = set(initial_facts)
    fired = set()               # 이미 발화한 규칙
    round_no = 0
    while True:
        round_no += 1
        new_facts = set()
        for name, rule in rules.items():
            if name in fired:
                continue
            if set(rule["if"]) <= facts:   # 조건 충족?
                new_facts.add(rule["then"])
                fired.add(name)
                print(f"[{round_no}회전] {name} 발화: "
                      f"→ {rule['then']}")
        if not (new_facts - facts):        # 종료 조건
            print(f"[{round_no}회전] 새 사실 없음 — 종료")
            return facts
        facts |= new_facts

**핵심 포인트**
- `set(rule["if"]) <= facts` 한 줄이 "조건이 **모두** 사실인가"의 검사다(부분집합 연산).
- **종료 조건**: 한 회전에서 새 사실이 하나도 안 나오면 멈춘다 — 이것이 없으면 무한 루프가 된다.
- 같은 규칙이 두 번 발화하지 않도록 `fired`로 기록한다.

### [셀 3] 추론 실행 — 이 동물은 무엇인가? 📖

In [4]:
# 관찰된 초기 사실 (본문 4.4 IF-THEN 손추적과 동일)
observed = ["털이 있다", "고기를 먹는다",
            "황갈색이다", "검은 줄무늬가 있다"]

print("초기 사실:", ", ".join(observed), "\n")
final = forward_chain(RULES, observed)

animals = {"호랑이", "치타", "기린", "조류"}
conclusion = final & animals
print("\n최종 결론:", conclusion or "판정 불가")

초기 사실: 털이 있다, 고기를 먹는다, 황갈색이다, 검은 줄무늬가 있다 

[1회전] R1 발화: → 포유류
[2회전] R5 발화: → 육식동물
[3회전] R8 발화: → 호랑이
[4회전] 새 사실 없음 — 종료

최종 결론: {'호랑이'}


**핵심 포인트**
- 유도 사슬: 털이 있다 → **포유류**(R1) → 고기를 먹는다와 결합해 **육식동물**(R5) → 무늬와 결합해 **호랑이**(R8). 본문 4.4 IF-THEN 추론의 손추적 진행표와 줄 단위로 같다.
- 관찰 사실 4개에서 결론까지 규칙이 **자동으로 연쇄**되는 것이 추론 엔진의 힘이다.

기대 출력: `[1회전] R1 발화 → 포유류`, `[2회전] R5 발화 → 육식동물`, `[3회전] R8 발화 → 호랑이`, 최종 결론 `{'호랑이'}`

실패 시 대처: 결론이 나오지 않으면 발화 로그의 마지막 규칙을 보고, 초기 사실의 **철자가 규칙 조건과 정확히 일치**하는지 확인한다(규칙 기반 시스템의 취약점 그 자체를 체험하는 셈이다).

### [보조 1] 여러 동물 일괄 판정

In [5]:
CASES = {
    "사례 A": ["털이 있다", "고기를 먹는다",
               "황갈색이다", "검은 줄무늬가 있다"],
    "사례 B": ["우유를 먹인다", "발굽이 있다",
               "목이 길다", "다리가 길다"],
    "사례 C": ["깃털이 있다"],
    "사례 D": ["털이 있다", "황갈색이다",
               "검은 점무늬가 있다"],   # 육식 근거 없음
}
animals = {"호랑이", "치타", "기린", "조류"}
print(f"{'사례':<6} {'결론':<8} 유도된 중간 사실")
for label, facts in CASES.items():
    import io, contextlib
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        final = forward_chain(RULES, facts)
    concl = final & animals
    mid = final - set(facts) - concl
    print(f"{label:<6} {str(concl or '판정 불가'):<8}"
          f" {mid or '-'}")

사례     결론       유도된 중간 사실
사례 A   {'호랑이'}  {'포유류', '육식동물'}
사례 B   {'기린'}   {'포유류', '유제류'}
사례 C   {'조류'}   -
사례 D   판정 불가    {'포유류'}


사례 D는 점무늬가 있어도 "고기를 먹는다"·"송곳니" 관찰이 없어 육식동물이 유도되지 않아 **판정 불가**다 — 지식의 공백이 추론의 공백이 된다(4.5 지식 획득 병목 예고).

### [보조 2] 후방 연쇄 미니 데모 — 목표에서 거꾸로

In [6]:
def backward_chain(rules, facts, goal, depth=0):
    """목표가 성립하는지 역방향으로 검증한다."""
    indent = "  " * depth
    if goal in facts:
        print(f"{indent}✓ '{goal}' — 관찰된 사실")
        return True
    for name, rule in rules.items():
        if rule["then"] == goal:
            print(f"{indent}? '{goal}' ← {name}의 "
                  f"조건을 검증")
            if all(backward_chain(rules, facts, c,
                                  depth + 1)
                   for c in rule["if"]):
                print(f"{indent}✓ '{goal}' — {name} 성립")
                return True
    print(f"{indent}✗ '{goal}' — 입증 실패")
    return False

facts = set(CASES["사례 A"])
print("목표: 이 동물은 호랑이인가?\n")
backward_chain(RULES, facts, "호랑이")

목표: 이 동물은 호랑이인가?

? '호랑이' ← R8의 조건을 검증
  ? '육식동물' ← R5의 조건을 검증
    ? '포유류' ← R1의 조건을 검증
      ✓ '털이 있다' — 관찰된 사실
    ✓ '포유류' — R1 성립
    ✓ '고기를 먹는다' — 관찰된 사실
  ✓ '육식동물' — R5 성립
  ✓ '황갈색이다' — 관찰된 사실
  ✓ '검은 줄무늬가 있다' — 관찰된 사실
✓ '호랑이' — R8 성립


True

전방 연쇄가 "사실에서 출발해 결론을 넓혀 가는" 방식이라면, 후방 연쇄는 "**목표를 세우고 필요한 조건을 거슬러 확인**"한다(4.4 IF-THEN 추론의 대칭 구도). 검증할 가설이 정해져 있을 때는 후방 연쇄가 불필요한 발화를 줄인다.

### [심화 1] 규칙 교체 — 나만의 미니 전문가 시스템 (연습문제 심화 직결)

In [7]:
# 연습문제 심화: 다른 도메인의 규칙 10개를 설계해
# 같은 엔진으로 돌려 보자. 엔진 수정은 필요 없다!
MY_RULES = {
    # 예: 학사 규정 (조건·결론의 어휘를 통일할 것)
    "S1": {"if": ["전공 60학점 이수"],
           "then": "전공 요건 충족"},
    "S2": {"if": ["전공 요건 충족", "총 130학점 이수",
                  "졸업논문 통과"], "then": "졸업 가능"},
    # TODO: 규칙을 10개까지 채워 보자
}
my_facts = ["전공 60학점 이수", "총 130학점 이수",
            "졸업논문 통과"]
forward_chain(MY_RULES, my_facts)

[1회전] S1 발화: → 전공 요건 충족
[2회전] S2 발화: → 졸업 가능
[3회전] 새 사실 없음 — 종료


{'전공 60학점 이수', '전공 요건 충족', '졸업 가능', '졸업논문 통과', '총 130학점 이수'}

---
## 마무리

- **지식(규칙)과 추론(엔진)의 분리**가 규칙 기반 시스템의 핵심 설계다 — 규칙만 바꾸면 다른 도메인의 전문가 시스템이 된다([심화 1]).
- 전방 연쇄는 사실 → 결론, 후방 연쇄는 목표 → 조건. 상황(데이터 주도 vs 가설 검증)에 따라 방향을 고른다.
- 철자 하나로 추론이 끊기는 취약함, 관찰이 없으면 판정 불가가 되는 공백 — 규칙 기반 접근의 한계(4.5)를 몸으로 확인했다.

**연습문제 연계**: [응용] 전방·후방 연쇄 유도 과정 추적은 [보조 2]로 자가 검증, [심화] 도메인 규칙 설계는 [심화 1] 뼈대에서 수행한다.

**다음 실습**: [실습 5-1] 나이브 베이즈 스팸 분류 (`ch05/lab-05-01_naive-bayes-spam.ipynb`)